In [3]:
import pandas as pd
import requests
from datetime import datetime, timedelta
import os

def load_noaa_events(csv_path='noaa_21-25_with_huc_08.csv'):
    """Loads NOAA dataset, standardizes 5-digit FIPS codes, and parses datetimes."""
    df = pd.read_csv(csv_path)
    
    # Construct 5-digit county FIPS
    df['FIPS_5'] = (
        df['STATE_FIPS'].astype(str).str.zfill(2) + 
        df['CZ_FIPS'].astype(str).str.zfill(3)
    )
    
    # Parse dates to datetime objects
    df['BEGIN_DT'] = pd.to_datetime(df['BEGIN_DATE_TIME'])
    df['END_DT'] = pd.to_datetime(df['END_DATE_TIME'])
    return df

def query_fema_claims_multi_fips(fips_list, start_date, end_date, date_buffer_days=14):
    """Queries OpenFEMA v2 NFIP claims across all impacted counties in the episode date window."""
    base_url = "https://www.fema.gov/api/open/v2/FimaNfipClaims"
    
    search_start = (start_date - timedelta(days=2)).strftime('%Y-%m-%d')
    search_end = (end_date + timedelta(days=date_buffer_days)).strftime('%Y-%m-%d')
    
    fips_conditions = " or ".join([f"countyCode eq '{fips}'" for fips in fips_list])
    
    fema_filter = (
        f"({fips_conditions}) and "
        f"dateOfLoss ge {search_start}T00:00:00.000Z and "
        f"dateOfLoss le {search_end}T23:59:59.000Z"
    )
    
    params = {
        "$filter": fema_filter,
        "$top": 10000
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=15)
        response.raise_for_status()
        records = response.json().get('FimaNfipClaims', [])
        return pd.DataFrame(records)
    except Exception as e:
        print(f"Error querying OpenFEMA API: {e}")
        return pd.DataFrame()

def query_usgs_hwms(min_lat, max_lat, min_lon, max_lon, start_date, end_date, date_buffer_days=30):
    """Queries USGS STN API and filters by bounding box and episode date window."""
    base_url = "https://stn.wim.usgs.gov/STNServices/HWMs.json"
    
    try:
        response = requests.get(base_url, timeout=15)
        response.raise_for_status()
        hwms = response.json()
        
        if not hwms:
            return pd.DataFrame()
            
        df_hwm = pd.DataFrame(hwms)
        
        # Map correct latitude / longitude field names in USGS STN JSON
        lat_col = 'latitude_dd' if 'latitude_dd' in df_hwm.columns else ('latitude' if 'latitude' in df_hwm.columns else None)
        lon_col = 'longitude_dd' if 'longitude_dd' in df_hwm.columns else ('longitude' if 'longitude' in df_hwm.columns else None)
        
        if not lat_col or not lon_col:
            print("Warning: Could not identify coordinate columns in USGS response.")
            return pd.DataFrame()

        # Parse survey / flag date safely
        date_col = None
        for col in ['flagDate', 'surveyDate', 'approvalDate']:
            if col in df_hwm.columns:
                date_col = col
                break
                
        if date_col:
            df_hwm['date_parsed'] = pd.to_datetime(df_hwm[date_col], errors='coerce')

        # Spatial filter with 0.1 degree padding (~10km)
        lat_pad, lon_pad = 0.1, 0.1
        mask_spatial = (
            (df_hwm[lat_col] >= (min_lat - lat_pad)) & 
            (df_hwm[lat_col] <= (max_lat + lat_pad)) & 
            (df_hwm[lon_col] >= (min_lon - lon_pad)) & 
            (df_hwm[lon_col] <= (max_lon + lon_pad))
        )
        
        # Temporal filter
        search_start = start_date - timedelta(days=2)
        search_end = end_date + timedelta(days=date_buffer_days)
        
        if 'date_parsed' in df_hwm.columns:
            mask_temporal = (
                (df_hwm['date_parsed'] >= search_start) & 
                (df_hwm['date_parsed'] <= search_end)
            )
            df_filtered = df_hwm[mask_spatial & mask_temporal]
        else:
            df_filtered = df_hwm[mask_spatial]
            
        return df_filtered
        
    except Exception as e:
        print(f"Error querying USGS STN API: {e}")
        return pd.DataFrame()

def run_pipeline():
    df = load_noaa_events()
    
    print("==================================================")
    print(" NOAA Episode -> OpenFEMA & USGS HWM Pipeline")
    print("==================================================")
    
    episodes_summary = df.groupby('NEW_EPISODE_ID').agg(
        event_types=('EVENT_TYPE', lambda x: ', '.join(x.unique())),
        counties=('CZ_NAME', lambda x: ', '.join(x.unique())),
        event_count=('EVENT_ID', 'count')
    ).reset_index()
    
    print("\nSample Episodes from CSV:\n")
    print(episodes_summary[['NEW_EPISODE_ID', 'event_types', 'counties', 'event_count']].head(10).to_string(index=False))
    print("\n--------------------------------------------------")
    
    user_input = input("\nEnter the NEW_EPISODE_ID you want to inspect (or press Enter for default '191899_0'): ").strip()
    if not user_input:
        user_input = "191899_0"
        
    episode_rows = df[df['NEW_EPISODE_ID'] == user_input]
    if episode_rows.empty:
        print(f"NEW_EPISODE_ID '{user_input}' not found in the dataset.")
        return
        
    # Extract metadata
    fips_list = episode_rows['FIPS_5'].unique().tolist()
    county_names = episode_rows['CZ_NAME'].unique().tolist()
    huc8_names = episode_rows['NAME'].unique().tolist()
    
    min_date = episode_rows['BEGIN_DT'].min()
    max_date = episode_rows['END_DT'].max()
    
    min_lat = episode_rows['BEGIN_LAT'].min()
    max_lat = episode_rows['END_LAT'].max()
    min_lon = episode_rows['BEGIN_LON'].min()
    max_lon = episode_rows['END_LON'].max()
    
    print("\n--------------------------------------------------")
    print(f"Selected Episode Summary:")
    print(f" • Episode ID:    {user_input}")
    print(f" • Event Types:   {', '.join(episode_rows['EVENT_TYPE'].unique())}")
    print(f" • Counties ({len(fips_list)}): {', '.join(county_names)} (FIPS: {', '.join(fips_list)})")
    print(f" • HUC-8 Basins:  {', '.join(huc8_names)}")
    print(f" • Date Window:   {min_date.strftime('%Y-%m-%d %H:%M')} to {max_date.strftime('%Y-%m-%d %H:%M')}")
    print(f" • Bounding Box:  Lat [{min_lat:.3f}, {max_lat:.3f}], Lon [{min_lon:.3f}, {max_lon:.3f}]")
    print("--------------------------------------------------\n")
    
    # --- 1. OPENFEMA QUERY ---
    print("Searching OpenFEMA for matching claims across all impacted counties...")
    claims_df = query_fema_claims_multi_fips(
        fips_list=fips_list, 
        start_date=min_date, 
        end_date=max_date
    )
    
    if claims_df.empty:
        print("No FEMA NFIP claims found for these counties around this episode window.\n")
    else:
        print(f"Found {len(claims_df)} matching FEMA claim(s)!\n")
        cols_to_show = [c for c in ['dateOfLoss', 'countyCode', 'amountPaidOnBuildingClaim', 'amountPaidOnContentsClaim'] if c in claims_df.columns]
        print(claims_df[cols_to_show].head(10).to_string(index=False))
        
        if 'amountPaidOnBuildingClaim' in claims_df.columns:
            tot_bldg = claims_df['amountPaidOnBuildingClaim'].sum()
            tot_cnt = claims_df['amountPaidOnContentsClaim'].sum() if 'amountPaidOnContentsClaim' in claims_df.columns else 0
            print("\nFinancial Totals for Matching Episode Claims:")
            print(f" • Building Payouts: ${tot_bldg:,.2f}")
            print(f" • Contents Payouts: ${tot_cnt:,.2f}")
            print(f" • Total Payouts:    ${(tot_bldg + tot_cnt):,.2f}")
            
        export_claims = input("\nWould you like to export these FEMA claims to a CSV file? (y/n): ").strip().lower()
        if export_claims in ['y', 'yes']:
            filename = f"fema_claims_episode_{user_input.replace('/', '_')}.csv"
            claims_df.to_csv(filename, index=False)
            print(f"✓ Saved {len(claims_df)} FEMA claims to '{filename}'!")

    print("\n--------------------------------------------------")
    
    # --- 2. USGS HWM QUERY ---
    print("Searching USGS STN for matching High-Water Marks...")
    hwms_df = query_usgs_hwms(
        min_lat=min_lat, max_lat=max_lat,
        min_lon=min_lon, max_lon=max_lon,
        start_date=min_date, end_date=max_date
    )
    
    if hwms_df.empty:
        print("No matching USGS High-Water Marks found for this bounding box/timeframe.\n")
    else:
        print(f"Found {len(hwms_df)} matching USGS High-Water Mark(s)!\n")
        cols_hwm = [c for c in ['hwmID', 'eventName', 'latitude_dd', 'longitude_dd', 'elevFt', 'hwmQualityName', 'hwmTypeName'] if c in hwms_df.columns]
        print(hwms_df[cols_hwm].head(10).to_string(index=False))
        
        export_hwms = input("\nWould you like to export these USGS High-Water Marks to a CSV file? (y/n): ").strip().lower()
        if export_hwms in ['y', 'yes']:
            filename = f"usgs_hwms_episode_{user_input.replace('/', '_')}.csv"
            hwms_df.to_csv(filename, index=False)
            print(f"✓ Saved {len(hwms_df)} USGS HWM records to '{filename}'!")

if __name__ == "__main__":
    run_pipeline()

 NOAA Episode -> OpenFEMA & USGS HWM Pipeline

Sample Episodes from CSV:

NEW_EPISODE_ID event_types             counties  event_count
      157601_0       Flood            VAN BUREN            1
      158463_0 Flash Flood JEFFERSON, VAN BUREN            4
      158466_0 Flash Flood                 LINN            1
      159463_0 Flash Flood       LUCAS, WAPELLO            3
      159795_0 Flash Flood            ALLAMAKEE            1
      160825_0 Flash Flood                STORY            2
      161012_0 Flash Flood            VAN BUREN            1
      161088_0 Flash Flood                WORTH            4
      161208_0 Flash Flood           DES MOINES            2
      161640_0 Flash Flood                 IOWA            1

--------------------------------------------------

--------------------------------------------------
Selected Episode Summary:
 • Episode ID:    191899_0
 • Event Types:   Flood
 • Counties (11): LYON, SIOUX, OSCEOLA, DICKINSON, CHEROKEE, O'BRIEN, PLYM